## Experiment 3: Event detection ResNet-18 - CNN evaluation

In [ ]:
import os
import torch
import numpy as np
import torch.nn as nn
import seaborn as sns
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
TEST_DATA_DIR = "/home/sermomon/PROJECTS/msr-das-cnn-resnet-train001/run001/test"
MODEL_PATH = "/home/sermomon/PROJECTS/msr-das-cnn-resnet-train001/run001/train_results/best.pth"
OUTPUT_DIR = "/home/sermomon/PROJECTS/msr-das-cnn-resnet-train001/run001/test_results"

IMG_SIZE = (681, 1028)
BATCH_SIZE = 4
RESNET_VARIANT = 18

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
test_transforms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
def load_test_data(test_dir, batch_size):
    """
    Load test dataset
    
    Parameters:
    -----------
    test_dir : str
        Path to test folder containing subfolders 0 and 1
    batch_size : int
        Batch size for DataLoader
    
    Returns:
    --------
    test_loader : DataLoader
        Test data loader
    class_names : list
        Class names
    """
    test_dataset = datasets.ImageFolder(
        root=test_dir,
        transform=test_transforms
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )
    
    print(f"\n{'='*60}")
    print(f"TEST DATA")
    print(f"{'='*60}")
    print(f"Test directory: {test_dir}")
    print(f"Classes detected: {test_dataset.classes}")
    print(f"Total samples: {len(test_dataset)}")
    
    # Count samples per class
    class_counts = {}
    for _, label in test_dataset.samples:
        class_name = test_dataset.classes[label]
        class_counts[class_name] = class_counts.get(class_name, 0) + 1
    
    for class_name, count in class_counts.items():
        print(f"  Class {class_name}: {count} samples")
    
    print(f"{'='*60}\n")
    
    return test_loader, test_dataset.classes

In [ ]:
_RESNET_VARIANTS = {
    18:  models.resnet18,
    34:  models.resnet34,
    50:  models.resnet50,
    101: models.resnet101,
    152: models.resnet152,
}

def load_model(model_path, num_classes, variant=18, device='cpu'):
    """
    Load trained ResNet model
    
    Parameters:
    -----------
    model_path : str
        Path to saved model weights
    num_classes : int
        Number of classes
    variant : int
        ResNet variant (18, 34, 50, 101, 152)
    device : torch.device
        Device to load model on
    
    Returns:
    --------
    model : torch.nn.Module
        Loaded model
    """
    if variant not in _RESNET_VARIANTS:
        raise ValueError(f"Variant {variant} not supported. Choose from {list(_RESNET_VARIANTS.keys())}")
    
    # Create model architecture
    model = _RESNET_VARIANTS[variant](weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    
    # Load weights
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    
    print(f"{'='*60}")
    print(f"MODEL LOADED")
    print(f"{'='*60}")
    print(f"Model: ResNet-{variant}")
    print(f"Weights: {model_path}")
    print(f"Classes: {num_classes}")
    print(f"Device: {device}")
    print(f"{'='*60}\n")
    
    return model

In [ ]:
def evaluate_model(model, test_loader, device, class_names):
    """
    Evaluate model on test set
    
    Parameters:
    -----------
    model : torch.nn.Module
        Trained model
    test_loader : DataLoader
        Test data loader
    device : torch.device
        Device
    class_names : list
        Class names
    
    Returns:
    --------
    results : dict
        Dictionary with predictions, labels, and metrics
    """
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    print("Evaluating model on test set...")
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            
            # Predictions
            probs = torch.softmax(outputs, dim=1)
            preds = outputs.argmax(dim=1).cpu().numpy()
            
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    # Calculate metrics
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)
    
    # Per-class metrics
    precision_per_class = precision_score(all_labels, all_preds, average=None, zero_division=0)
    recall_per_class = recall_score(all_labels, all_preds, average=None, zero_division=0)
    f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)
    
    results = {
        'predictions': all_preds,
        'labels': all_labels,
        'probabilities': all_probs,
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': cm,
        'precision_per_class': precision_per_class,
        'recall_per_class': recall_per_class,
        'f1_per_class': f1_per_class,
        'class_names': class_names
    }
    
    return results

In [ ]:
def print_results(results):
    """
    Print evaluation results
    
    Parameters:
    -----------
    results : dict
        Results dictionary from evaluate_model
    """
    print(f"\n{'='*60}")
    print(f"TEST RESULTS")
    print(f"{'='*60}")
    print(f"Overall Metrics:")
    print(f"  Accuracy  : {results['accuracy']:.4f}")
    print(f"  Precision : {results['precision']:.4f}")
    print(f"  Recall    : {results['recall']:.4f}")
    print(f"  F1-Score  : {results['f1']:.4f}")
    
    print(f"\nPer-Class Metrics:")
    for i, class_name in enumerate(results['class_names']):
        print(f"  Class {class_name}:")
        print(f"    Precision: {results['precision_per_class'][i]:.4f}")
        print(f"    Recall:    {results['recall_per_class'][i]:.4f}")
        print(f"    F1-Score:  {results['f1_per_class'][i]:.4f}")
    
    print(f"\nConfusion Matrix:")
    print(f"  Classes: {results['class_names']}")
    print(results['confusion_matrix'])
    print(f"{'='*60}\n")

In [ ]:
def plot_confusion_matrix(results, output_dir):
    """
    Plot and save confusion matrix
    
    Parameters:
    -----------
    results : dict
        Results dictionary
    output_dir : str
        Output directory
    """
    cm = results['confusion_matrix']
    class_names = results['class_names']
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Plot with seaborn for better aesthetics
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Count'}, ax=ax)
    
    ax.set_xlabel('Predicted', fontsize=14)
    ax.set_ylabel('True', fontsize=14)
    ax.set_title('Confusion Matrix - Test Set', fontsize=16)
    
    plt.tight_layout()
    
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, 'confusion_matrix_test.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Confusion matrix saved: {save_path}")
    plt.show()

In [ ]:
def plot_normalized_confusion_matrix(results, output_dir):
    """
    Plot normalized confusion matrix (percentages)
    
    Parameters:
    -----------
    results : dict
        Results dictionary
    output_dir : str
        Output directory
    """
    cm = results['confusion_matrix']
    class_names = results['class_names']
    
    # Normalize
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Percentage'}, ax=ax)
    
    ax.set_xlabel('Predicted', fontsize=14)
    ax.set_ylabel('True', fontsize=14)
    ax.set_title('Normalized Confusion Matrix - Test Set', fontsize=16)
    
    plt.tight_layout()
    
    save_path = os.path.join(output_dir, 'confusion_matrix_normalized_test.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Normalized confusion matrix saved: {save_path}")
    plt.show()

In [ ]:
def save_results_to_file(results, output_dir):
    """
    Save results to text file
    
    Parameters:
    -----------
    results : dict
        Results dictionary
    output_dir : str
        Output directory
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, 'test_results.txt')
    
    with open(save_path, 'w') as f:
        f.write("="*60 + "\n")
        f.write("TEST RESULTS\n")
        f.write("="*60 + "\n\n")
        
        f.write("Overall Metrics:\n")
        f.write(f"  Accuracy  : {results['accuracy']:.4f}\n")
        f.write(f"  Precision : {results['precision']:.4f}\n")
        f.write(f"  Recall    : {results['recall']:.4f}\n")
        f.write(f"  F1-Score  : {results['f1']:.4f}\n\n")
        
        f.write("Per-Class Metrics:\n")
        for i, class_name in enumerate(results['class_names']):
            f.write(f"  Class {class_name}:\n")
            f.write(f"    Precision: {results['precision_per_class'][i]:.4f}\n")
            f.write(f"    Recall:    {results['recall_per_class'][i]:.4f}\n")
            f.write(f"    F1-Score:  {results['f1_per_class'][i]:.4f}\n")
        
        f.write(f"\nConfusion Matrix:\n")
        f.write(f"Classes: {results['class_names']}\n")
        f.write(str(results['confusion_matrix']) + "\n")
        f.write("="*60 + "\n")
    
    print(f"Results saved to: {save_path}")

In [ ]:
print(f"\nDevice: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}\n")

# 1. Load test data
test_loader, class_names = load_test_data(TEST_DATA_DIR, BATCH_SIZE)
num_classes = len(class_names)

# 2. Load model
model = load_model(MODEL_PATH, num_classes, variant=RESNET_VARIANT, device=device)

# 3. Evaluate
results = evaluate_model(model, test_loader, device, class_names)

# 4. Print results
print_results(results)

# 5. Save results
save_results_to_file(results, OUTPUT_DIR)

# 6. Plot confusion matrices
plot_confusion_matrix(results, OUTPUT_DIR)
plot_normalized_confusion_matrix(results, OUTPUT_DIR)

print(f"\n✓ Evaluation completed!")
print(f"Results saved in: {OUTPUT_DIR}")